# Tuchanka design notebook

Design, preview, and export gcode for the Tuchanka (Prusa XL 5-tool) printer.

Run cells top to bottom with shift+enter. If you change a cell, re-run it and all cells below it.

In [ ]:
import fullcontrol as fc
from fullcontrol.devices.personal.tuchanka.setup import starting_steps, ending_steps, base_settings

In [ ]:
# printer / startup parameters

design_name = 'my_design'
initial_tool = 0
tools_used = [0]
first_layer_temps = [215.0] * 5   # per-tool, °C
idle_temps        = [70.0]  * 5   # per-tool standby, °C
bed_temp          = 60.0
print_area        = (20, 20, 180, 180)  # (x_min, y_min, x_max, y_max) mm

# set heat_soak=True or nozzle_clean=True to enable those startup stages
startup = starting_steps(
    initial_tool=initial_tool,
    tools_used=tools_used,
    first_layer_temps=first_layer_temps,
    idle_temps=idle_temps,
    bed_temp=bed_temp,
    print_area=print_area,
    heat_soak=False,
    nozzle_clean=False,
)

In [ ]:
# design parameters

EW = 0.4  # extrusion width (mm)
EH = 0.2  # extrusion height / layer height (mm)
initial_z = EH * 0.6  # slight squish for first layer adhesion
print_speed = 1000  # mm/min

In [ ]:
# H design — single layer, filled, centered on the XL bed (360x360mm)
#
# Letter dimensions (all mm):
#   height:       100
#   total width:   80  (left bar 15 | gap 50 | right bar 15)
#   crossbar height: 15, centered vertically
#
# Bed center: (180, 180)

letter_h      = 100   # total height
letter_w      = 80    # total width
stroke_w      = 15    # thickness of each vertical bar and the crossbar
cx, cy        = 180.0, 180.0  # center of bed

x0 = cx - letter_w / 2   # 140 — left edge of left bar
x1 = x0 + stroke_w        # 155 — right edge of left bar
x2 = cx + letter_w / 2 - stroke_w  # 205 — left edge of right bar
x3 = cx + letter_w / 2    # 220 — right edge of right bar

y_bot = cy - letter_h / 2  # 130
y_top = cy + letter_h / 2  # 230

cb_bot = cy - stroke_w / 2  # 172.5 — crossbar bottom
cb_top = cy + stroke_w / 2  # 187.5 — crossbar top

z = initial_z


def fill_rect(x_left, x_right, y_bottom, y_top, z, ew):
    """Raster-fill a rectangle with parallel X-direction lines spaced EW apart."""
    steps = []
    y = y_bottom + ew / 2
    row = 0
    while y <= y_top + ew / 2:
        if row % 2 == 0:
            steps.append(fc.Extruder(on=False))
            steps.append(fc.Point(x=x_left,  y=y, z=z))
            steps.append(fc.Extruder(on=True))
            steps.append(fc.Point(x=x_right, y=y, z=z))
        else:
            steps.append(fc.Extruder(on=False))
            steps.append(fc.Point(x=x_right, y=y, z=z))
            steps.append(fc.Extruder(on=True))
            steps.append(fc.Point(x=x_left,  y=y, z=z))
        y += ew
        row += 1
    return steps


design_steps = []
design_steps += fill_rect(x0, x1, y_bot, y_top, z, EW)   # left bar
design_steps += fill_rect(x2, x3, y_bot, y_top, z, EW)   # right bar
design_steps += fill_rect(x1, x2, cb_bot, cb_top, z, EW) # crossbar

In [ ]:
# preview the design (startup gcode steps are excluded — they're not geometry)

fc.transform(design_steps, 'plot', fc.PlotControls(style='line', zoom=0.7))

# for a more realistic preview showing actual extrusion widths, uncomment:
# fc.transform(design_steps, 'plot', fc.PlotControls(style='tube', zoom=0.7, initialization_data={'extrusion_width': EW, 'extrusion_height': EH}))

In [ ]:
# generate and save gcode

shutdown = ending_steps(travel_speed=8000, present_print=True)
steps = startup + design_steps + shutdown

gcode_controls = fc.GcodeControls(
    printer_name='custom',
    save_as=design_name,
    initialization_data=base_settings(
        print_speed=print_speed,
        extrusion_width=EW,
        extrusion_height=EH,
        nozzle_temp=first_layer_temps[initial_tool],
        bed_temp=bed_temp,
    ),
)
gcode = fc.transform(steps, 'gcode', gcode_controls)

---
*Tuchanka notebook — Petricore Labs*